# ClinVar RAG — query & summarize

Pipeline: **ingest** → **index** → **search** → **summarize** (search and summarize run in this notebook).

Prerequisites: `python src/ingest_clinvar.py` then `python src/build_clinvar_index.py`. EDA: `clinvar_eda.ipynb`.

In [176]:
# Natural-language query
query = "Which variants cause benign tumors?"


In [177]:
# Optional: install dependencies if needed
# !pip install chromadb sentence-transformers transformers huggingface_hub accelerate

In [178]:
# Imports
from pathlib import Path
import sys

from sentence_transformers import SentenceTransformer


In [179]:
# Project paths and shared RAG helpers
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root / "src"))

from rag_search import search
from rag_summary import load_llm, summarize
from rag_export import append_rag_report


## Search

Embed query → retrieve chunks from Chroma → rank variants → build LLM context.

In [180]:
# Retrieval limits
TOP_K_CHUNKS = 20  # chunk pool from Chroma; used to rank variants (not sent to LLM)
TOP_K_RESULTS = 3  # top variants by best-chunk distance; LLM gets 1 full parquet row each

In [181]:
# Paths, embedding model, search config
data_processed = project_root / "data/processed"
chroma_dir = project_root / "data/chroma_db"

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
CLINVAR_PARQUET = data_processed / "clinvar_rag.parquet"
CLINVAR_COLLECTION = "clinvar_chunks"
CLINVAR_RECORD_FIELDS = [
    "VariationID", "Name", "GeneSymbol", "Chromosome", "Start", "Stop",
    "Type", "ClinicalSignificance", "ReviewStatus", "PhenotypeList",
    "PhenotypeIDS", "Assembly", "HGNC_ID", "NumberSubmitters", "LastEvaluated",
]

# Maps Chroma chunk metadata and parquet columns to rag_search.search()
SEARCH_CONFIG = {
    "collection_name": CLINVAR_COLLECTION,  # Chroma collection to query
    "parquet_path": CLINVAR_PARQUET,        # full records joined onto retrieved chunks
    "group_key": "variation_id",            # metadata field: collapse chunks to one variant
    "record_id_column": "VariationID",      # parquet column used to look up full records
    "record_id_meta_key": "variation_id",   # chunk metadata field holding the same ID
    "record_fields": CLINVAR_RECORD_FIELDS, # parquet columns included in LLM context
    "section_header_template": "### Result {rank} — variation_id={variation_id}, gene={gene}",  # per-result header; placeholders from metadata
    "hit_fields": [("variation_id", "variation_id"), ("gene", "gene")],  # metadata keys printed for each hit
    "full_record_label": "Full ClinVar record",       # label above the joined parquet row
    "selected_label": "variant record(s) for context",  # verbose log when loading records
    "record_id_cast": int,  # cast metadata ID before parquet lookup (VariationID is int)
}


In [182]:
embed_model = SentenceTransformer(EMBED_MODEL)
# returns ranked hits + context string for the LLM
result = search(
    query,
    SEARCH_CONFIG,
    top_k_chunks=TOP_K_CHUNKS,
    top_k_results=TOP_K_RESULTS,
    chroma_dir=chroma_dir,
    embed_model=embed_model,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Collection clinvar_chunks: 11591 chunks

RETRIEVAL
Query: Which variants cause benign tumors?
Chroma returned the top 20 matching chunks (20 chunks across 20 unique record(s)).
Top 3 record(s) for the LLM: among proteins in this pool, rank by each protein's best (lowest-distance) chunk, then take the top 3. Chunks are used for ranking only — the LLM gets one full parquet row per selected record.
(17 chunk(s) in the pool belong to proteins outside the top 3.)

TOP 3 RECORD(S) (ranked by best-matching chunk per record)
Preview below is the closest chunk only (truncated to 500 chars).

1. variation_id=412157 gene=DICER1 best_chunk=summary dist=0.4154 (1 retrieved chunk(s))
Variation: 412157
Type: summary
gene: DICER1
pathogenicity: Likely benign
impact: DICER1-related tumor predisposition. Hereditary cancer-predisposing syndrome. Euthyroid goiter. Global developmental delay - lung cysts - overgrowth - Wilms tumor syndrome. Rhabdomyosarcoma, embryonal, 2;Euthyroid goiter;Pleuropulmonary bl

## Summarize


In [183]:
SYSTEM_PROMPT = (
    "You are a clinical genomics assistant. "
    "Write a brief summary in plain English sentences that answers the user's query using ONLY the ClinVar records provided. "
    "Always cite the ClinVar VariationID and Name in your answer. "
    "Do not invent genes, significance, phenotypes, or other facts not present in the provided text. "
    "If information is missing, say so briefly."
)

# SYSTEM_PROMPT_1 = (
#     "You are a clinical genomics assistant. Summarize ClinVar variant search results "
#     "for the user's query using ONLY the full records provided. "
#     "Be concise, accurate, and structured. For each relevant variant, mention gene, "
#     "variant name, clinical significance, phenotypes, review status, and variation ID. "
#     "If information is missing, say so. Do not invent facts."
# )

Load summarizer → generate answer from retrieved context only.

- **local:** downloads once → cached under `~/.cache/huggingface/`; 0.5B ~2GB RAM.
- **openai:** calls `gpt-4o-mini` via API; set `OPENAI_API_KEY` in `.env`.

In [184]:
# Summarization backend: "local" or "openai"
SUMMARIZER_BACKEND = "openai"
# Local model (SUMMARIZER_BACKEND == "local")
# Needs ~8GB+ RAM: LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
# Needs ~4GB+ RAM: LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# OpenAI model (SUMMARIZER_BACKEND == "openai")
API_MODEL = "gpt-4o-mini"
MAX_NEW_TOKENS = 1024
REPORT_PATH = project_root / "reports/rag_log.md"
CSV_LOG_PATH = project_root / "data/reports/rag_log.csv"

In [185]:
if SUMMARIZER_BACKEND == "local":
    llm_model, llm_tokenizer = load_llm(
        LLM_MODEL,
        model=globals().get("llm_model"),
        tokenizer=globals().get("llm_tokenizer"),
    )
else:
    llm_model, llm_tokenizer = None, None


In [186]:
summary = summarize(
    query,
    result["context"],
    llm_model,
    llm_tokenizer,
    SYSTEM_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
    backend=SUMMARIZER_BACKEND,
    api_model=API_MODEL,
)
print("=" * 60)
print(f"Query: {query}\n")
print(summary)

_ = append_rag_report(
    REPORT_PATH,
    query,
    result,
    summary,
    source_name="clinvar",
    summarizer_backend=SUMMARIZER_BACKEND,
    model_name=API_MODEL if SUMMARIZER_BACKEND == "openai" else LLM_MODEL,
    csv_path=CSV_LOG_PATH,
)


Summarized via OpenAI: gpt-4o-mini
Query: Which variants cause benign tumors?

The following variants are classified as likely benign and are associated with DICER1-related tumor predisposition:

1. VariationID: 412157, Name: NM_177438.3(DICER1):c.128C>T (p.Thr43Met)
2. VariationID: 412179, Name: NM_177438.3(DICER1):c.4412C>T (p.Pro1471Leu)
3. VariationID: 315107, Name: NM_177438.3(DICER1):c.3269+14G>A

These variants may be linked to benign tumors, but specific details about the tumors are not provided in the records.
Appended RAG report to rag_log.md
Appended RAG CSV row to rag_log.csv
